# Introducción a TensorFlow

TensorFlow es la librería de Google para machine learning y deep learning. Su objeto base es el **tensor** (un arreglo multidimensional que puede vivir en CPU o GPU) y su API de alto nivel se llama **Keras**: permite apilar capas como bloques de construcción sin escribir el gradiente a mano.

Este lab es un tutorial mínimo y ejecutable: crear capas `Dense`, aplicarlas como funciones, entender las formas 1D vs 2D, distinguir tensores de arreglos numpy, armar un `Sequential` y ver qué hacen `compile`, `fit` y `predict` (cada uno a fondo en su momento).

### La capa Dense: una neurona del Lab 02 empaquetada

Una capa `Dense` es una capa totalmente conectada: **cada neurona de salida escucha a todas las entradas**. Es exactamente la fórmula del lab anterior ($a = g(Wx + b)$), pero como objeto reutilizable.

Dos argumentos mandan:

- **`units`**: cuántas neuronas tiene la capa (cuántos números de salida produce).
- **`activation`**: qué función $g$ aplica cada neurona (`"sigmoid"`, `"relu"`, etc.).

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow {tf.__version__} | Keras {keras.__version__}")

# Paso 1: Crear una capa Dense de 3 neuronas con activación sigmoide
layer1 = layers.Dense(units=3, activation="sigmoid")
print(f"units = {layer1.units} | activation = {layer1.activation.__name__}")
print(f"La capa espera entradas de dimensión: {layer1.units} salidas por ejemplo")

I0000 00:00:1793600468.461660    8700 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow 2.21.0 | Keras 3.15.1
units = 3 | activation = sigmoid
La capa espera entradas de dimensión: 3 salidas por ejemplo


### La capa como función: `a = layer1(x_input)`

Una capa ya construida se usa como una función: recibe un vector de entrada y devuelve el vector de activaciones. Con 4 features de entrada y `units=3`, el resultado siempre tiene 3 números, uno por neurona. (Nótese que con entradas grandes sin escalar y pesos aleatorios la sigmoide se satura cerca de 0 o 1: el mismo aviso de escalamiento del Lab 05.)

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

# Paso 2: Aplicar la capa a un ejemplo (lote de 1 x 4 features)
layer1 = layers.Dense(units=3, activation="sigmoid")
x_input = np.array([[200.0, 15.0, 30.0, 1.0]])
print(f"forma de x_input: {x_input.shape} (lotes, features)")
a = layer1(x_input)
print(f"forma de salida: {tuple(a.shape)} (1 ejemplo, 3 activaciones)")
print(f"activaciones: {np.round(a.numpy().ravel(), 3)}")

# ¿Y si el vector viene en 1D? Keras trabaja por lotes: hay que remodelar
x_plano = np.array([200.0, 15.0, 30.0, 1.0])
print(f"vector 1D: {x_plano.shape} -> remodelado: {x_plano.reshape(1, -1).shape}")
try:
    a_malo = layer1(x_plano)
    print(f"con 1D sale: {tuple(a_malo.shape)}")
except Exception as e:
    print(f"con 1D falla: {type(e).__name__}: {str(e)[:120]}")
print(f"con 2D sale: {tuple(layer1(x_plano.reshape(1, -1)).shape)}")

forma de x_input: (1, 4) (lotes, features)
forma de salida: (1, 3) (1 ejemplo, 3 activaciones)
activaciones: [0. 0. 1.]
vector 1D: (4,) -> remodelado: (1, 4)
con 1D falla: ValueError: Input 0 with name 'None' of layer 'dense_1' is incompatible with the layer: expected min_ndim=2, found ndim=1. Full shap
con 2D sale: (1, 3)


### Vectores 1D vs 2D: mismos números, distinta forma

Un vector 1D de forma `(4,)` y una matriz de forma `(1, 4)` guardan los mismos 4 números, pero Keras los trata distinto: **siempre espera lotes en 2D**, con forma `(lotes, features)`. Un solo ejemplo es un lote de tamaño 1, o sea `(1, 4)`.

Por eso el paso estándar antes de predecir es remodelar: de `(4,)` a `(1, 4)` con `reshape(1, -1)`. El `-1` le dice a NumPy que complete la dimensión sola. Regla práctica: si un error de Keras habla de formas incompatibles, lo primero a revisar es si falta la dimensión del lote.

### Tensores vs arreglos de numpy

Un `tf.Tensor` y un `np.ndarray` se parecen (arreglos multidimensionales con forma y tipo de dato), pero no son lo mismo:

- El tensor vive donde TensorFlow decida (CPU o GPU) y está pensado para derivarse en el entrenamiento; el arreglo numpy vive en CPU y es para cálculo general.
- Los tensores tienen `dtype` explícito y controlado (`float32` por defecto) y no se modifican por asignación directa como un arreglo.
- Se convierten en ambas direcciones: `tf.constant()` y `tf.convert_to_tensor()` crean tensores, y `.numpy()` devuelve el arreglo equivalente.

In [3]:
import numpy as np
import tensorflow as tf

# Paso 3: Tensores desde listas y desde numpy, y de vuelta
t_lista = tf.constant([1.0, 2.0, 3.0, 4.0])
print(f"desde lista: {t_lista.shape} {t_lista.dtype}")
arr = np.array([[200.0, 15.0, 30.0, 1.0]])
t_arr = tf.convert_to_tensor(arr)
print(f"desde numpy: {tuple(t_arr.shape)} {t_arr.dtype}")
de_vuelta = t_arr.numpy()
print(f"de vuelta a numpy: {type(de_vuelta).__name__} {de_vuelta.shape}")

desde lista: (4,) <dtype: 'float32'>
desde numpy: (1, 4) <dtype: 'float64'>
de vuelta a numpy: ndarray (1, 4)


### Simplificar todo: `model = Sequential([...])`

En vez de crear capas sueltas y llamarlas una por una, `Sequential` las apila en orden: la salida de cada capa alimenta a la siguiente automáticamente. Es la red completa del Lab 01 en dos líneas: una hidden layer de 3 neuronas y una output layer de 1.

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential, layers

# Paso 4: La red completa en dos líneas
model = Sequential([
    layers.Dense(units=3, activation="sigmoid"),
    layers.Dense(units=1, activation="sigmoid"),
])
model.build(input_shape=(None, 4))
model.summary()
x_input = np.array([[200.0, 15.0, 30.0, 1.0]])
print(f"salida del modelo: {model(x_input).numpy().ravel().round(3)} (1 número)")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 3)              │            15 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19 (76.00 B)

 Trainable params: 19 (76.00 B)

 Non-trainable params: 0 (0.00 B)

salida del modelo: [0.673] (1 número)


### `compile`, `fit` y `predict`: qué hace cada uno (sin fondo, por ahora)

- **`model.compile(...)`**: declara *cómo* se va a entrenar — qué optimizador ajusta los pesos y con qué función de costo se mide el error. No entrena nada todavía.
- **`model.fit(x, y, ...)`**: entrena de verdad — pasa los datos varias veces (épocas) y ajusta los pesos con gradiente descendiente. Aquí sí se aprende.
- **`model.predict(new_x)`**: usa el modelo ya entrenado con datos nuevos. No aprende, solo calcula la salida.

El siguiente bloque los muestra en acción con un ajuste mínimo, solo para ver el flujo completo; el fondo de cada uno viene en su lab.

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential, layers

# Paso 5: Flujo mínimo completo con datos de juguete
tf.random.set_seed(7)
np.random.seed(7)
x = np.array([[0.0], [1.0], [2.0], [3.0]], dtype=np.float32)
y = np.array([[0.0], [0.0], [1.0], [1.0]], dtype=np.float32)
mini = Sequential([layers.Dense(units=1, activation="sigmoid")])
mini.compile(optimizer="sgd", loss="binary_crossentropy")
mini.fit(x, y, epochs=200, verbose=0)
print("compile: declarado | fit: entrenado")
nuevo = np.array([[1.5]], dtype=np.float32)
print(f"predict({nuevo.ravel()[0]}): P(clase 1) = {float(mini.predict(nuevo, verbose=0).ravel()[0]):.3f}")

compile: declarado | fit: entrenado
predict(1.5): P(clase 1) = 0.714


### Cierre

TensorFlow aporta los tensores y Keras las piezas: `Dense` para capas, `Sequential` para apilarlas, y el trío `compile` (cómo), `fit` (entrenar) y `predict` (usar). Los siguientes labs abren cada pieza: funciones de activación a fondo, cómo se entrena de verdad y cómo se evalúa.